# Riboflavin ligand parameterization for constant-pH MD

This notebook presents the **final project workflow** used to prepare multiple riboflavin protonation states for constant-pH molecular dynamics (CpHMD) with multisite λ-dynamics.

It consolidates the useful steps from several exploratory notebooks created during the project. Paths have been converted to repository-relative paths and lab-provided MSLD helper implementations have been moved into `scripts/brooks_lab/` so that provenance is clear.

**Project-specific work represented here:** preprocessing, CGenFF workflow automation, structural alignment, atom-name/mapping validation, reference-state selection, charge-renormalization integration, CHARMM topology/parameter assembly, and build-input generation.

**Lab-provided methodology/infrastructure:** the MSLD maximum-common-substructure and charge-renormalization helper routines imported from `scripts/brooks_lab/`.


## 1. Imports and repository paths

The notebook assumes it is run either from the repository root or from the `notebooks/` directory. All paths below are derived from the repository root rather than from a user-specific home directory.


In [ ]:
from pathlib import Path
import os
import re
import shutil
import subprocess
import sys

import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdFMCS
from openbabel import pybel


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "preprocessing").exists() and (candidate / "scripts").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)  # MSLD helper files use paths stored in text outputs.

INPUT_MOL2_DIR = REPO_ROOT / "preprocessing/input_data/mol2_epik"
CGENFF_DIR = REPO_ROOT / "preprocessing/cgenff_output"
CONVERTED_SDF_DIR = REPO_ROOT / "preprocessing/structure_conversion/converted_sdf"
ALIGNED_SDF_DIR = REPO_ROOT / "preprocessing/alignment/sdf_aligned"
ALIGNED_MOL2_DIR = REPO_ROOT / "preprocessing/alignment/mol2_aligned"
MCS_DIR = REPO_ROOT / "preprocessing/mcs"
CHARGE_DIR = REPO_ROOT / "preprocessing/charge_renormalization"
CHARMM_SETUP_DIR = REPO_ROOT / "preprocessing/charmm_setup"

for directory in [CGENFF_DIR, CONVERTED_SDF_DIR, ALIGNED_SDF_DIR, ALIGNED_MOL2_DIR,
                  MCS_DIR, CHARGE_DIR, CHARMM_SETUP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)


## 2. Generate CGenFF parameter streams

Each Epik-generated MOL2 protonation state was submitted to CGenFF. This step requires the cluster environment used during the project (`module load cgenff`), so it is disabled by default for repository browsing.


In [ ]:
RUN_CGENFF = False

if RUN_CGENFF:
    for mol2_path in sorted(INPUT_MOL2_DIR.glob("*.mol2")):
        output_str = CGENFF_DIR / f"{mol2_path.stem}.str"
        command = f"module load cgenff && cgenff -a < {mol2_path} > {output_str}"
        try:
            subprocess.run(command, shell=True, executable="/bin/bash", check=True)
            print(f"Processed {mol2_path.name}")
        except subprocess.CalledProcessError as exc:
            print(f"Failed processing {mol2_path.name}: {exc}")
else:
    print("CGenFF generation skipped. Set RUN_CGENFF=True in an environment with CGenFF available.")


## 3. Align protonation states and assign consistent atom names

The MSLD MCS workflow requires the ligand states to be spatially aligned and to contain unique atom names. The code below aligns all SDF structures on their maximum common substructure, writes aligned SDFs, converts them to MOL2, and assigns element-based unique atom names (`C1`, `C2`, `O1`, ...).

The aligned MOL2 directory also contains the corresponding CGenFF RTF/PRM files required by the downstream MSLD helper code.


In [ ]:
def assign_unique_atom_names(mol):
    counts = {}
    for atom in mol.GetAtoms():
        element = atom.GetSymbol()
        counts[element] = counts.get(element, 0) + 1
        atom.SetProp("_TriposAtomName", f"{element}{counts[element]}")


sdf_files = sorted(CONVERTED_SDF_DIR.glob("*.sdf"))
molecules = []
for sdf_path in sdf_files:
    mol = Chem.MolFromMolFile(str(sdf_path), removeHs=False)
    if mol is None:
        print(f"WARNING: failed to load {sdf_path.name}; skipping")
    else:
        molecules.append((sdf_path.name, mol))

if not molecules:
    raise RuntimeError("No SDF structures were loaded.")

reference_name, reference_mol = molecules[0]
mcs_result = rdFMCS.FindMCS(
    [mol for _, mol in molecules],
    threshold=1.0,
    completeRingsOnly=True,
    ringMatchesRingOnly=True,
)
mcs_mol = Chem.MolFromSmarts(mcs_result.smartsString)
reference_match = reference_mol.GetSubstructMatch(mcs_mol)

if not reference_match:
    raise RuntimeError("Reference molecule does not contain the identified MCS.")

print("MCS SMARTS:", mcs_result.smartsString)

for filename, mol in molecules:
    mol_match = mol.GetSubstructMatch(mcs_mol)
    if not mol_match:
        print(f"WARNING: {filename} does not contain the MCS; skipping")
        continue

    if not mol.GetNumConformers():
        AllChem.EmbedMolecule(mol)
        AllChem.UFFOptimizeMolecule(mol)

    atom_map = list(zip(mol_match, reference_match))
    rmsd = AllChem.AlignMol(mol, reference_mol, atomMap=atom_map)
    Chem.MolToMolFile(mol, str(ALIGNED_SDF_DIR / filename))
    print(f"Aligned {filename}: core RMSD = {rmsd:.3f} Å")

for sdf_path in sorted(ALIGNED_SDF_DIR.glob("*.sdf")):
    mol = Chem.MolFromMolFile(str(sdf_path), removeHs=False)
    if mol is None:
        print(f"WARNING: failed to reload {sdf_path.name}; skipping")
        continue

    assign_unique_atom_names(mol)
    temporary_mol = ALIGNED_SDF_DIR / "_temporary_conversion.mol"
    Chem.MolToMolFile(mol, str(temporary_mol))
    pybel_mol = next(pybel.readfile("mol", str(temporary_mol)))

    for atom_index, atom in enumerate(pybel_mol.atoms):
        atom.name = mol.GetAtomWithIdx(atom_index).GetProp("_TriposAtomName")

    output_mol2 = ALIGNED_MOL2_DIR / f"{sdf_path.stem}.mol2"
    pybel_mol.write("mol2", str(output_mol2), overwrite=True)
    temporary_mol.unlink(missing_ok=True)

    # MSLD helper code expects the matching RTF beside each aligned MOL2 file.
    for suffix in (".rtf", ".prm"):
        source = CGENFF_DIR / f"{sdf_path.stem}{suffix}"
        if source.exists():
            shutil.copy2(source, ALIGNED_MOL2_DIR / source.name)

print("Aligned structures written to", ALIGNED_MOL2_DIR)


## 4. Validate atom mappings across states

A subtle source of failure was inconsistent atom naming after inverse mapping. I wrote a separate validation utility (`scripts/check_mol2_atom_consistency.py`) to flag atom names assigned to different coordinates across state-specific MOL2 files.

This is the code sample highlighted in my DESRES application. Running it from the repository root:

```bash
python scripts/check_mol2_atom_consistency.py preprocessing/alignment/mol2_aligned
```


In [ ]:
from scripts.check_mol2_atom_consistency import find_conflicting_atoms

conflicts = find_conflicting_atoms(ALIGNED_MOL2_DIR)
if conflicts:
    for atom_name, file_coordinates in sorted(conflicts.items()):
        print(atom_name, file_coordinates)
else:
    print("No conflicting atom mappings found.")


## 5. MSLD maximum-common-substructure analysis

The next stage uses the **Brooks Lab MSLD helper implementation**. The helper itself is kept under `scripts/brooks_lab/` rather than embedded in this notebook so that the distinction between lab infrastructure and my project-specific workflow is explicit.


In [ ]:
BROOKS_HELPERS = REPO_ROOT / "scripts/brooks_lab"
if str(BROOKS_HELPERS) not in sys.path:
    sys.path.insert(0, str(BROOKS_HELPERS))

from msld_mcs import MsldMCS

mol2_bases = [
    f"preprocessing/alignment/mol2_aligned/{path.stem}"
    for path in sorted(ALIGNED_MOL2_DIR.glob("riboflavin_*.mol2"))
]
mol_list_file = ALIGNED_MOL2_DIR / "mol_list.txt"
mol_list_file.write_text("\n".join(mol2_bases) + "\n")

mcs_results_file = MCS_DIR / "mcs_results.txt"
MsldMCS(
    molfile=str(mol_list_file.relative_to(REPO_ROOT)),
    mcsout=str(mcs_results_file.relative_to(REPO_ROOT)),
    cutoff=0.8,
    debug=True,
)


## 6. Select a reference ligand state

CGenFF RTF files were parsed to compare the net charge and maximum atom penalty for each protonation state. Among states with the middle net charge represented in the set, the state with the lowest maximum CGenFF penalty was selected as the reference ligand and written into the MCS results file.


In [ ]:
ligand_info = []

for rtf_path in sorted(CGENFF_DIR.glob("riboflavin_*.rtf")):
    charge = 0.0
    penalties = []
    for line in rtf_path.read_text().splitlines():
        if line.strip().startswith("RESI"):
            parts = line.split()
            if len(parts) > 2:
                try:
                    charge = float(parts[2])
                except ValueError:
                    pass
        elif line.strip().startswith("ATOM"):
            match = re.search(r"! *penalty *= *([0-9.]+)", line)
            if match:
                penalties.append(float(match.group(1)))

    ligand_info.append({
        "lig_name": rtf_path.stem,
        "charge": charge,
        "max_penalty": max(penalties) if penalties else float("inf"),
    })

charges = sorted({item["charge"] for item in ligand_info})
if not charges:
    raise RuntimeError("No CGenFF RTF files were found.")

middle_charge = charges[len(charges) // 2]
candidates = [item for item in ligand_info if item["charge"] == middle_charge]
base_state = min(candidates, key=lambda item: item["max_penalty"])
print("Selected reference state:", base_state)

lines = mcs_results_file.read_text().splitlines(keepends=True)
reference_base = f"preprocessing/alignment/mol2_aligned/{base_state['lig_name']}"
with mcs_results_file.open("w") as handle:
    for line in lines:
        if line.startswith("REFLIG"):
            handle.write(f"REFLIG {reference_base}\n")
        else:
            handle.write(line)


## 7. Charge renormalization

Charge renormalization is performed with the **Brooks Lab MSLD helper routine** separated into `scripts/brooks_lab/msld_crn.py`. The project-specific portion below assembles the mappings between ligand states and their CGenFF files and invokes the helper.


In [ ]:
from msld_crn import MsldCRN

in_frag = {
    mol2_path.stem: str(mol2_path.relative_to(REPO_ROOT))
    for mol2_path in sorted(INPUT_MOL2_DIR.glob("riboflavin_*.mol2"))
}

an_core = {}
for ligand_name in sorted(in_frag):
    an_core[ligand_name] = {
        "str": str((CGENFF_DIR / f"{ligand_name}.str").relative_to(REPO_ROOT)),
        "prm": str((CGENFF_DIR / f"{ligand_name}.prm").relative_to(REPO_ROOT)),
    }

MsldCRN(
    mcsout=str(mcs_results_file.relative_to(REPO_ROOT)),
    outdir=str(CHARGE_DIR.relative_to(REPO_ROOT)),
    inFrag=in_frag,
    AnCore=an_core,
    verbose=True,
)


## 8. Assemble the CHARMM topology and parameter files

The charge-renormalization output contains one core RTF and state-specific patch RTFs. I combined these into a single topology and concatenated the per-state parameter files into a combined parameter file for the ligand build.


In [ ]:
from scripts.combine_rtf_files import combine_rtf_files

combined_rtf = CHARMM_SETUP_DIR / "riboflavin_combined_patch/combined_patches.rtf"
combine_rtf_files(
    core_rtf_path=CHARGE_DIR / "core.rtf",
    patches_dir=CHARGE_DIR,
    output_rtf_path=combined_rtf,
)

combined_prm = CGENFF_DIR / "ligand_combined.prm"
parameter_files = [
    path for path in sorted(CGENFF_DIR.glob("riboflavin_*.prm"))
    if path.name != combined_prm.name
]

with combined_prm.open("w") as output:
    for prm_path in parameter_files:
        output.write(f"! Start of {prm_path.name}\n")
        output.write(prm_path.read_text())
        output.write(f"\n! End of {prm_path.name}\n\n")

print(f"Combined {len(parameter_files)} parameter files into {combined_prm}")


## 9. Generate the CHARMM ligand-build input

The final preprocessing step writes a compact CHARMM input file that reads the combined topology/parameters and the charge-renormalized large-ligand PDB, then builds and writes the ligand PSF/PDB.


In [ ]:
rtf_path = combined_rtf.relative_to(REPO_ROOT)
prm_path = combined_prm.relative_to(REPO_ROOT)
pdb_path = (CHARGE_DIR / "large_lig.pdb").relative_to(REPO_ROOT)

charmm_input = f"""read rtf card name {rtf_path}
read param card name {prm_path}
read sequence pdb name {pdb_path}
read coor pdb name {pdb_path}

generate LIG setup
ic param
ic build

write psf card name LIG.psf
write coor pdb name LIG.pdb

stop
"""

build_input_path = CHARMM_SETUP_DIR / "build_ligand.inp"
build_input_path.write_text(charmm_input)
print("Wrote", build_input_path)


## Notes on reproducibility

This repository is a **research code sample**, not a standalone software package. Several steps require software or environment modules used in the Brooks Lab/HPC workflow (for example CGenFF, CHARMM/pyCHARMM, Open Babel, and the MSLD/ALF infrastructure). The committed preprocessing outputs are retained to document the sequence of transformations used during the project.

The original exploratory notebooks remain available through the repository's Git history; this notebook is the curated representation of the final workflow.
